In [2]:
import dask.array as da
from aicsimageio import AICSImage
import os
import re
from tifffile import imread
import napari
from napari.settings import get_settings
settings=get_settings()
settings.application.ipy_interactive=True
import tifffile
import xml.etree.ElementTree as ET
import pandas as pd
import zarr
from itertools import cycle
import numpy as np

from skimage.transform import resize

In [6]:
main_dir=rf'D:\thu\HTAN\images'
analysis_dir=os.getcwd()
nst_dir=os.path.join(main_dir,'NST')
ilc_dir=os.path.join(main_dir,'ILC')

In [ ]:
list_slide_antibodies={}
total_list_antibodies =[]
for file in os.listdir(nst_dir):
    path=os.path.join(nst_dir,file)
    with tifffile.TiffFile(path) as tif:
        root=ET.fromstring(tif.ome_metadata)

        list_antibodies=[]
        for element in root.iter():
            if element.tag.endswith('Channel'):
                print(element.tag)
                print(element.attrib)
                print("-"*20)
                list_antibodies.append(element.attrib['Name'])
                total_list_antibodies.append(element.attrib['Name'])    
    list_slide_antibodies[f'{file}']=list_antibodies

{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:0', 'SamplesPerPixel': '1', 'Name': 'DNA'}
--------------------
{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:1', 'SamplesPerPixel': '1', 'Name': 'Autofluorescence-488nm'}
--------------------
{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:2', 'SamplesPerPixel': '1', 'Name': 'Autofluorescence-555nm'}
--------------------
{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:3', 'SamplesPerPixel': '1', 'Name': 'Autofluorescence-647nm'}
--------------------
{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:4', 'SamplesPerPixel': '1', 'Name': 'DNA (2)'}
--------------------
{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:5', 'SamplesPerPixel': '1', 'Name': 'Control-488nm'}
--------------------
{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel
{'ID': 'Channel:0:6', 'S

In [21]:
num_pRB_file = 0
for file in os.listdir(nst_dir):
    path=os.path.join(nst_dir,file)
    with tifffile.TiffFile(path) as tif:
        root = ET.fromstring(tif.ome_metadata)
        for element in root.iter():
            if element.tag.endswith('Channel'):
                if element.attrib['Name']=='Rb (pS807; pS811)':
                    num_pRB_file +=1
                    

In [22]:
def get_image_by_marker(tif_path:str,
                        marker:str)-> np.ndarray:
    with tifffile.TiffFile(tif_path) as tif:
        root=ET.fromstring(tif.ome_metadata)
        for element in root.iter():
                if element.tag.endswith('Channel'): # this stays consistent with all tiff files
                    biomarker_name=element.attrib['Name']
                    if biomarker_name==f'{marker}':
                        channel=int(element.attrib['ID'].split(":")[-1])
                        return imread(tif_path)[channel]

In [24]:
# Image shape aare all (2672, 2672)

In [ ]:
test_slide='OHSU_TMA1_010-C8.ome.tif'
test_path=os.path.join(main_dir,test_slide)
test_image=imread(test_path)

test_slide_2="TNP-TMA-28_F3-ILC-3B.ome.tif"
test_path_2=os.path.join(main_dir,test_slide_2)
test_image_2=imread(test_path_2)

s1 = AICSImage(test_path).get_image_dask_data("CYX")
s2 = AICSImage(test_path_2).get_image_dask_data("CYX")

print(s1.shape, s2.shape)  # check this first

(40, 2672, 2672) (55, 5051, 5051)


In [46]:

test_slide='TNP-TMA-2-Scene-A07.ome.tif'
test_path=os.path.join(main_dir,test_slide)
test_image=imread(test_path)

test_slide_2="TNP-TMA-2-Scene-B01.ome.tif"
test_path_2=os.path.join(main_dir,test_slide_2)
test_image_2=imread(test_path_2)

s1 = AICSImage(test_path).get_image_dask_data("CYX")
s2 = AICSImage(test_path_2).get_image_dask_data("CYX")

print(s1.shape, s2.shape)  # check this first

(55, 5724, 7582) (55, 7578, 5728)


In [32]:
def resize_image(image:np.ndarray, target_size: tuple)-> np.ndarray:
    new_image=resize(image, output_shape=target_size,
                    preserve_range=True, anti_aliasing=True).astype(image.dtype)
    return new_image

In [48]:
cav_image_1=get_image_by_marker(test_path,'CAV1')
cav_image_2=get_image_by_marker(test_path_2,'CAV1')

dapi_image_1=get_image_by_marker(test_path,'DAPI9')
dapi_image_2=get_image_by_marker(test_path_2,'DAPI9')

viewer=napari.Viewer()
viewer.add_image(cav_image_1,name='cav1_image1')
viewer.add_image(cav_image_2,name='cav1_image2')
viewer.add_image(dapi_image_1,name='dapi1')
viewer.add_image(dapi_image_2,name='dapi2')

<Image layer 'dapi2' at 0x14785ff3a90>

In [ ]:
viewer=napari.Viewer()
viewer.add_image(test_montage,name=f'{test_slide}_{test_slide_2}')
viewer.add_image(test_dapi,name='Dapi')

viewer.add_image(er_image_2,name='er_2')
viewer.add_image(er_image_2_resized,name='er_2_resize')
viewer.add_image(dapi_image_2,name='dapi_2')
viewer.add_image(dapi_image_2_resized,name='dapi_image_2_resized')

viewer.add_image(er_image_1,name='er_1')
viewer.add_image(dapi_image_1,name='dapi_1')


In [ ]:

er_image_1=get_image_by_marker(test_path,'ER')
er_image_2=get_image_by_marker(test_path_2,'ER')
er_image_2_resized=resize_image(er_image_2,target_size=(2672, 2672))

dapi_image_1=get_image_by_marker(test_path,'DNA (6)')
dapi_image_2=get_image_by_marker(test_path_2,'DAPI2')
dapi_image_2_resized=resize_image(dapi_image_2,target_size=(2672, 2672))

gap=np.zeros((2672,50))
test_montage=np.concatenate([er_image_1,gap,er_image_2_resized],axis=1)
test_dapi=np.concatenate([dapi_image_1,gap,dapi_image_2_resized],axis=1)

viewer=napari.Viewer()
viewer.add_image(test_montage,name=f'{test_slide}_{test_slide_2}')
viewer.add_image(test_dapi,name='Dapi')

viewer.add_image(er_image_2,name='er_2')
viewer.add_image(er_image_2_resized,name='er_2_resize')
viewer.add_image(dapi_image_2,name='dapi_2')
viewer.add_image(dapi_image_2_resized,name='dapi_image_2_resized')

viewer.add_image(er_image_1,name='er_1')
viewer.add_image(dapi_image_1,name='dapi_1')

